# 2: Field Detection

In [1]:
import dill
import json

import numpy as np
import scipy
import networkx as nx
import pandas as pd
import seaborn as sns

from collections import Counter
from scipy.sparse import csr_matrix

In [2]:
processed_data_path = '../data/processed/'

lexicon_fn = processed_data_path + 'lexicon.json'
tes_fn = processed_data_path + 'tes.pkl'
colex_fn = processed_data_path + 'colex.pkl'
fields_fn = processed_data_path + 'fields.pkl'
merged_tes_fn = processed_data_path + 'merged_tes.pkl'
saturation_fn = processed_data_path + 'saturation.csv'

In [12]:
lexicon = json.load(open(lexicon_fn, 'r'))
tes = dill.load(open(tes_fn, 'rb'))
colex = dill.load(open(colex_fn, 'rb'))
merged_tes = dill.load(open(merged_tes_fn, 'rb'))

In [13]:
### UTILITY FUNCTIONS

def idx2lex(lexicon, selected):
    return np.array(lexicon)[selected]

def lex2idx(lexicon, selected):
    return [lexicon.index(s) for s in selected]

def fld2str(field):
    s = ''
    for w in field:
        s += w if len(s)==0 else '_' + w
    return s

def str2fld(field_string):
    return field_string.split('_')

def print_colex(lexicon, colex):
    rows = sorted(list(set(colex.nonzero()[0])))
    for i in rows:
        cols = colex.getrow(i).nonzero()[1]
        #print(i, cols)
        if len(cols) > 1:
            print(lexicon[i], ":", [(lexicon[j], int(colex[i,j])) for j in cols])

In [47]:
### LEXICAL FIELDS

def generate_fields(colex, k):
    graph = nx.Graph(colex)
    fields = list(map(list, nx.community.k_clique_communities(graph, k)))
    arcs = graph.edges

    for (a,b) in arcs:
        present = False
        for f in fields:
            if a in f and b in f:
                present = True
                break
        if not present:
            fields.append((a,b))
            
    #present = {i for f in fields for i in f}
    #for (a,b) in graph.edges:
    #    if not (a in present and b in present):
    #        if [b,a] not in fields:
    #            fields.append([a,b])

    return fields

def structure_fields(fields, tes, lexicon, saturation_threshold=1.0):
    flds = list()
    sat = 0
    tot = 0
    for field in fields:
        field = idx2lex(lexicon, field)
        lf = dict()
        
        for lang in tes:
            lf[lang] = dict()
            tot += 1
            if saturation_ratio(field, tes[lang]) >= saturation_threshold:
                sat += 1
                #lf[lang] = Counter([stem for seed in field if seed in tes[lang][0] for stem in tes[lang][0][seed]])

                # get stems & their counts for field in lang
                stems = dict()
                for seed in field:
                    if seed in tes[lang][0]:
                        for stem in tes[lang][0][seed]:
                            if stem in stems:
                                stems[stem] += len(tes[lang][0][seed][stem])
                            else:
                                stems[stem] = len(tes[lang][0][seed][stem])

                # remove stems with count=1
                lf[lang] = {s: c for s, c in stems.items() if c > 1}
        flds.append((field, lf))
    print("Obs:", sat, "/", tot)
    return flds

def saturation_ratio(field, langtes):
    count = 0
    for term in field:
        # add cleaning of langtes keys ?
        # rethink about that
        if term in langtes[0]:
            count += 1
    return count / len(field)

def generate_saturation_dataframe(lexicon, fields, tes):
    saturation = {'lang': [], 'field': [], 'ratio': []}
    for lang in tes:
        for field in fields:
            field = idx2lex(lexicon, field)
            ratio = saturation_ratio(field, tes[lang])
            saturation['lang'].append(lang)
            saturation['field'].append(field_to_str(field))
            saturation['ratio'].append(ratio)
    return pd.DataFrame(saturation)

In [15]:
### PARAMETERS
SATURATION_THRESHOLD = 1

In [50]:
fields = generate_fields(colex, 3)
for f in fields:
    print(idx2lex(lexicon, list(f)))

['remove' 'take' 'get' 'come' 'reach' 'leave' 'bring' 'arrive' 'carry']
['kill' 'hit' 'beat']
['beautiful' 'nice' 'good']
['say' 'call' 'language' 'happen' 'talk' 'become' 'build' 'speak' 'make'
 'do' 'tell' 'story' 'put']
['beginning' 'start' 'begin']
['many' 'large' 'elder' 'old' 'big']
['child' 'boy' 'guy' 'man' 'son']
['hold' 'catch' 'take']
['world' 'ground' 'country' 'land' 'earth']
['tomorrow' 'day' 'morning']
['time' 'day' 'today']
['dead' 'kill' 'die']
['listen' 'hear' 'feel']
['small' 'little' 'few' 'young' 'son']
['see' 'look' 'find' 'watch']
['go' 'pass' 'walk']
['understand' 'know' 'hear']
['person' 'people' 'human' 'man']
['lie' 'lay' 'put']
['stay' 'sit' 'live' 'settle']
['marriage' 'wedding' 'marry']
['way' 'path' 'road']
['thought' 'think' 'plan']
['stay' 'remain' 'leave']
['accept' 'agree']
['adult' 'old']
['afraid' 'frightened']
['afraid' 'scared']
['afternoon' 'day']
['afternoon' 'evening']
['alive' 'life']
['anger' 'angry']
['angry' 'heart']
['animal' 'hunt']
['arm

In [51]:
len(fields)

265

In [52]:
fields = structure_fields(arclics, merged_tes, lexicon, SATURATION_THRESHOLD)
dill.dump(fields, open(fields_fn, 'wb'))

Obs: 7339 / 12720


In [ ]:
saturation_df = generate_saturation_dataframe(lexicon, arclics, tes)
saturation_df.to_csv(saturation_fn)
saturation_df

In [ ]:
sns.histplot(saturation_df.ratio)

In [ ]:
langs_per_field = pd.DataFrame({"field": [field[0] for field in fields], "nb_lang": [len([lang for lang in field[1] if len(field[1][lang]) > 0]) for field in fields]})
langs_per_field.to_csv(processed_data_path + 'langsxfield.csv')
langs_per_field.sort_values(by="nb_lang")

In [ ]:
avg_terms_per_field = pd.DataFrame({"lang": tes.keys(), "avg_terms_per_field": [np.mean([len(field[1][lang]) for field in fields]) for lang in tes]})
avg_terms_per_field.to_csv(processed_data_path + 'avg_terms_per_field.csv')
avg_terms_per_field.sort_values(by="avg_terms_per_field", ascending=False)
#sns.barplot(data=avg_terms_per_field, x="lang", y="avg_terms_per_field")

In [ ]:
# connected components
g = nx.Graph(colex)
nx.connected_components(g)
[len(c) for c in sorted(nx.connected_components(g), key=len, reverse=True)]
len(max(nx.connected_components(g), key=len))